# Crypto Sentiment & Trader Behavior Analysis
**Platform:** Hyperliquid | **Sentiment:** Bitcoin Fear & Greed Index | **Period:** May 2023 – May 2025

---

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
import warnings
warnings.filterwarnings('ignore')

OUTPUTS = "./outputs"
import os; os.makedirs(OUTPUTS, exist_ok=True)
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.05)
plt.rcParams.update({"figure.dpi": 130, "savefig.bbox": "tight"})
print("Imports OK")

## 2. Load Data

In [ ]:
fg_raw   = pd.read_csv("./fear_greed_index.csv")
hist_raw = pd.read_csv("./historical_data.csv")

print("=== SHAPE ===")
print(f"Fear/Greed : {fg_raw.shape}")
print(f"Historical : {hist_raw.shape}")
print("\n=== COLUMNS ===")
print("FG  :", fg_raw.columns.tolist())
print("Hist:", hist_raw.columns.tolist())
print("\n=== MISSING VALUES ===")
print("FG  :", fg_raw.isnull().sum().to_dict())
print("Hist:", hist_raw.isnull().sum().to_dict())
print(f"\nFG dupes : {fg_raw.duplicated().sum()}")
print(f"Hist dupes: {hist_raw.duplicated().sum()}")

## 3. Data Cleaning

In [ ]:
# Fear/Greed: parse dates, derive binary sentiment
fg = fg_raw.copy()
fg['date'] = pd.to_datetime(fg['date'])
fg['sentiment_binary'] = fg['classification'].map(
    lambda x: 'Fear' if 'Fear' in x else ('Greed' if 'Greed' in x else 'Neutral'))
fg = fg[['date','value','classification','sentiment_binary']].sort_values('date')

# Historical: parse IST timestamp string
hist = hist_raw.copy()
hist['ts']   = pd.to_datetime(hist['Timestamp IST'], format='%d-%m-%Y %H:%M')
hist['date'] = hist['ts'].dt.normalize()

# Numeric coercions
for col in ['Execution Price','Size Tokens','Size USD','Closed PnL','Fee']:
    hist[col] = pd.to_numeric(hist[col], errors='coerce')

# Directional flags
long_dirs  = {'Open Long','Buy','Long > Short','Close Short'}
short_dirs = {'Open Short','Sell','Short > Long','Close Long'}
hist['is_long']    = hist['Direction'].isin(long_dirs).astype(int)
hist['is_short']   = hist['Direction'].isin(short_dirs).astype(int)
hist['is_closing'] = hist['Direction'].str.contains('Close|Settlement|Liquidat', na=False).astype(int)
hist['pnl_close']  = np.where(hist['is_closing']==1, hist['Closed PnL'], np.nan)

print(f"Hist date range: {hist['date'].min().date()} to {hist['date'].max().date()}")
print("Cleaning complete.")

## 4. Feature Engineering

In [ ]:
def safe_winrate(x):
    pos = (x > 0).sum(); tot = (x != 0).sum()
    return pos/tot if tot > 0 else np.nan

grp = hist.groupby('date')
daily = pd.DataFrame({
    'total_pnl':      grp['Closed PnL'].sum(),
    'median_pnl':     grp['Closed PnL'].median(),
    'pnl_std':        grp['Closed PnL'].std(),
    'win_rate':       grp['pnl_close'].apply(safe_winrate),
    'trade_count':    grp.size(),
    'avg_size_usd':   grp['Size USD'].mean(),
    'total_volume':   grp['Size USD'].sum(),
    'long_count':     grp['is_long'].sum(),
    'short_count':    grp['is_short'].sum(),
    'unique_traders': grp['Account'].nunique(),
    'avg_fee':        grp['Fee'].mean(),
}).reset_index()

daily['long_short_ratio']  = daily['long_count'] / (daily['short_count'] + 1e-9)
daily['avg_pnl_per_trade'] = daily['total_pnl'] / daily['trade_count']
daily['rolling_pnl_7d']    = daily['total_pnl'].rolling(7, min_periods=3).mean()

merged = daily.merge(fg, on='date', how='inner')
print(f"Merged: {len(merged)} days | {merged['date'].min().date()} to {merged['date'].max().date()}")
print("Sentiment dist:", merged['classification'].value_counts().to_dict())
merged.head(3)

## 5. Exploratory Analysis

In [ ]:
print("Daily PnL stats:")
print(merged['total_pnl'].describe().round(2))
print("\nSentiment value stats:")
print(merged['value'].describe().round(2))

## 6. Sentiment vs Performance

In [ ]:
bm = merged[merged['sentiment_binary'].isin(['Fear','Greed'])].copy()

perf = bm.groupby('sentiment_binary').agg(
    avg_daily_pnl     = ('total_pnl','mean'),
    median_daily_pnl  = ('total_pnl','median'),
    avg_pnl_per_trade = ('avg_pnl_per_trade','mean'),
    win_rate          = ('win_rate','mean'),
    pnl_volatility    = ('pnl_std','mean'),
    avg_trades_day    = ('trade_count','mean'),
    avg_size_usd      = ('avg_size_usd','mean'),
    days              = ('total_pnl','count'),
).round(3)
print(perf.T.to_string())

def max_drawdown(s):
    cs = s.cumsum(); return (cs - cs.cummax()).min()

print("\nDrawdown proxy by sentiment:")
print(bm.groupby('sentiment_binary')['total_pnl'].apply(max_drawdown).round(1).to_dict())

In [ ]:
# Chart 1: PnL by Sentiment
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle("Trader PnL by Bitcoin Sentiment", fontsize=14, fontweight='bold')
order5   = ['Extreme Fear','Fear','Neutral','Greed','Extreme Greed']
palette5 = ['#d62728','#ff7f0e','#9467bd','#2ca02c','#1f77b4']
mv = merged[merged['classification'].isin(order5)]

ax = axes[0]
means5 = mv.groupby('classification')['total_pnl'].mean().reindex(order5)
ax.bar(range(len(order5)), means5.values, color=palette5, edgecolor='k', linewidth=0.5)
ax.set_xticks(range(len(order5))); ax.set_xticklabels(order5, rotation=30, ha='right', fontsize=8)
ax.set_title("Avg Daily PnL by Sentiment Class"); ax.set_ylabel("Avg Total PnL (USD)")
ax.axhline(0, color='black', lw=0.8, ls='--')
for i,v in enumerate(means5.values):
    ax.text(i, v+means5.max()*0.02, f'${v:,.0f}', ha='center', fontsize=7)

ax = axes[1]
sns.boxplot(data=bm, x='sentiment_binary', y='total_pnl', order=['Fear','Greed'],
            palette=['#ff7f0e','#2ca02c'], ax=ax, width=0.5,
            flierprops=dict(marker='o', markersize=2, alpha=0.3))
ax.set_title("Daily PnL Distribution: Fear vs Greed"); ax.axhline(0, color='k', lw=0.8, ls='--')

ax = axes[2]
wr2 = bm.groupby('sentiment_binary')[['win_rate','avg_pnl_per_trade']].mean()
ax.bar(np.arange(2), wr2['win_rate'], 0.5, color=['#ff7f0e','#2ca02c'], edgecolor='k', alpha=0.85, label='Win Rate')
ax2b = ax.twinx()
ax2b.plot(np.arange(2), wr2['avg_pnl_per_trade'], 'D--', color='navy', ms=9, label='Avg PnL/Trade')
ax.set_xticks([0,1]); ax.set_xticklabels(['Fear','Greed'])
ax.set_title("Win Rate & Avg PnL per Trade"); ax.set_ylabel("Win Rate"); ax2b.set_ylabel("Avg PnL/Trade USD")
h1,l1=ax.get_legend_handles_labels(); h2,l2=ax2b.get_legend_handles_labels()
ax.legend(h1+h2, l1+l2, fontsize=8)
plt.tight_layout()
plt.savefig(f"{OUTPUTS}/pnl_by_sentiment.png")
plt.show()

In [ ]:
# Chart 2: Behavioral Metrics
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
fig.suptitle("Trader Behavior by Sentiment (Fear vs Greed)", fontsize=14, fontweight='bold')
for (col, title, ax) in [
    ('avg_size_usd','Avg Trade Size (USD)',axes[0,0]),
    ('trade_count','Avg Daily Trades',axes[0,1]),
    ('long_short_ratio','Long/Short Ratio',axes[1,0]),
    ('pnl_std','PnL Volatility (Std Dev)',axes[1,1])
]:
    vals = bm.groupby('sentiment_binary')[col].mean()
    colors = ['#ff7f0e' if s=='Fear' else '#2ca02c' for s in vals.index]
    ax.bar(vals.index, vals.values, color=colors, edgecolor='k')
    ax.set_title(title)
    if col=='long_short_ratio': ax.axhline(1, color='k', ls='--', lw=0.8)
    for i,v in enumerate(vals.values): ax.text(i, v*1.01, f"{v:.2f}", ha='center', fontsize=9)
plt.tight_layout()
plt.savefig(f"{OUTPUTS}/leverage_distribution.png")
plt.show()

In [ ]:
# Chart 3: Rolling PnL vs Sentiment Index
fig, ax1 = plt.subplots(figsize=(14, 5))
ax2 = ax1.twinx()
ax1.fill_between(merged['date'], merged['rolling_pnl_7d'].fillna(0), alpha=0.25, color='steelblue')
ax1.plot(merged['date'], merged['rolling_pnl_7d'], color='steelblue', lw=1.5, label='7d Rolling PnL')
ax2.plot(merged['date'], merged['value'], color='darkorange', lw=1, alpha=0.7, label='F&G Index')
ax2.axhline(50, color='gray', ls='--', lw=0.7)
ax1.set_ylabel("7-Day Rolling PnL (USD)", color='steelblue')
ax2.set_ylabel("Fear & Greed Index", color='darkorange')
ax1.set_title("7-Day Rolling Market PnL vs Bitcoin Fear & Greed Index", fontweight='bold')
h1,l1=ax1.get_legend_handles_labels(); h2,l2=ax2.get_legend_handles_labels()
ax1.legend(h1+h2, l1+l2, loc='upper left', fontsize=9)
plt.tight_layout()
plt.savefig(f"{OUTPUTS}/rolling_pnl_vs_sentiment.png")
plt.show()

In [ ]:
# Chart 4: Monthly Heatmap
month_map = {1:'Jan',2:'Feb',3:'Mar',4:'Apr',5:'May',6:'Jun',
             7:'Jul',8:'Aug',9:'Sep',10:'Oct',11:'Nov',12:'Dec'}
merged['month_name'] = merged['date'].dt.month.map(month_map)
month_order = [v for v in month_map.values() if v in merged['month_name'].unique()]
pivot = merged.pivot_table(index='month_name', columns='sentiment_binary',
                            values='total_pnl', aggfunc='mean').reindex(month_order)
fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(pivot, annot=True, fmt=".0f", cmap="RdYlGn", center=0, ax=ax,
            linewidths=0.4, cbar_kws={'label':'Avg Daily PnL (USD)'})
ax.set_title("Monthly Avg Daily PnL by Sentiment Regime", fontweight='bold')
plt.tight_layout()
plt.savefig(f"{OUTPUTS}/monthly_pnl_heatmap.png")
plt.show()

## 7. Trader Segmentation

In [ ]:
tg = hist.groupby('Account')
tstats = pd.DataFrame({
    'total_pnl':    tg['Closed PnL'].sum(),
    'avg_pnl':      tg['Closed PnL'].mean(),
    'pnl_std':      tg['Closed PnL'].std(),
    'trade_count':  tg.size(),
    'avg_size_usd': tg['Size USD'].mean(),
    'win_rate':     tg['pnl_close'].apply(safe_winrate),
    'total_volume': tg['Size USD'].sum(),
    'long_frac':    tg['is_long'].mean(),
    'active_days':  tg['date'].nunique(),
}).reset_index()

tstats['trades_per_day']  = tstats['trade_count'] / tstats['active_days'].clip(lower=1)
tstats['pnl_consistency'] = tstats['avg_pnl'] / (tstats['pnl_std'] + 1e-9)
tstats['prof_seg']        = np.where(tstats['total_pnl']>0,'Profitable','Unprofitable')
tstats['size_seg']        = pd.qcut(tstats['avg_size_usd'], q=3, labels=['Small','Medium','Large'])
tstats['freq_seg']        = pd.qcut(tstats['trades_per_day'], q=3,
                                     labels=['Infrequent','Moderate','Frequent'], duplicates='drop')

print("PROFITABILITY SEGMENTS:")
print(tstats.groupby('prof_seg').agg(
    count=('total_pnl','count'), avg_total_pnl=('total_pnl','mean'),
    avg_win_rate=('win_rate','mean'), avg_size=('avg_size_usd','mean'),
    avg_tpd=('trades_per_day','mean'),
).round(2).to_string())
print("\nSIZE SEGMENTS:")
print(tstats.groupby('size_seg')[['total_pnl','win_rate','trades_per_day']].mean().round(2).to_string())

In [ ]:
# KMeans Clustering — Behavioral Archetypes
feats = ['avg_pnl','pnl_std','trades_per_day','avg_size_usd','win_rate','long_frac']
ct = tstats.dropna(subset=feats).copy()
X = ct[feats].replace([np.inf,-np.inf], np.nan).fillna(ct[feats].median())
Xs = StandardScaler().fit_transform(X)
km = KMeans(n_clusters=4, random_state=42, n_init=10)
ct['cluster'] = km.fit_predict(Xs)

cg = ct.groupby('cluster')
max_pnl  = cg['avg_pnl'].mean().idxmax()
max_vol  = cg['avg_size_usd'].mean().idxmax()
max_freq = cg['trades_per_day'].mean().idxmax()
label_map = {max_pnl:'Elite Traders', max_vol:'High-Volume Gamblers', max_freq:'Scalpers'}
ct['archetype'] = ct['cluster'].map(lambda c: label_map.get(c,'Casual Traders'))

print("TRADER ARCHETYPES:")
print(ct.groupby('archetype')[['total_pnl','avg_pnl','trades_per_day','avg_size_usd','win_rate']].mean().round(2).to_string())

In [ ]:
# Chart 5: Trader Segments
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle("Trader Segmentation", fontsize=14, fontweight='bold')

ps = tstats.groupby('prof_seg')['avg_pnl'].mean()
axes[0].bar(ps.index, ps.values, color=['#d62728','#2ca02c'], edgecolor='k')
axes[0].set_title("Avg PnL/Trade: Profitable vs Unprofitable"); axes[0].axhline(0,color='k',lw=0.8,ls='--')

ss = tstats.groupby('size_seg')['total_pnl'].mean()
axes[1].bar(ss.index, ss.values, color=['#1f77b4','#ff7f0e','#d62728'], edgecolor='k')
axes[1].set_title("Avg Total PnL by Position Size"); axes[1].axhline(0,color='k',lw=0.8,ls='--')

arch = ct.groupby('archetype')['total_pnl'].mean().sort_values(ascending=False)
axes[2].bar(arch.index, arch.values, color=['#2ca02c','#1f77b4','#ff7f0e','#9467bd'], edgecolor='k')
axes[2].set_title("Avg Total PnL by Archetype")
axes[2].set_xticklabels(arch.index, rotation=15, ha='right', fontsize=8)
axes[2].axhline(0,color='k',lw=0.8,ls='--')

plt.tight_layout()
plt.savefig(f"{OUTPUTS}/trader_segments.png")
plt.show()

In [ ]:
# Chart 6: Archetype Scatter
fig, ax = plt.subplots(figsize=(9, 6))
arch_palette = {'Elite Traders':'#2ca02c','Scalpers':'#1f77b4',
                'High-Volume Gamblers':'#d62728','Casual Traders':'#9467bd'}
for aname, gdf in ct.groupby('archetype'):
    ax.scatter(gdf['trades_per_day'].clip(upper=200), gdf['avg_pnl'].clip(-500,1000),
               alpha=0.4, s=20, label=aname, color=arch_palette.get(aname,'gray'))
ax.axhline(0, color='k', lw=0.8, ls='--')
ax.set_xlabel("Trades per Day (capped 200)"); ax.set_ylabel("Avg PnL/Trade USD (capped +-500)")
ax.set_title("Trader Archetypes: Frequency vs Profitability", fontweight='bold')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig(f"{OUTPUTS}/archetype_scatter.png")
plt.show()

## 8. Key Insights

### Insight 1 — Fear drives volume, not quality
Fear days produce **2.5x higher aggregate PnL** ($39K vs $15.8K avg daily) but win rates are *lower* (27.5% vs 34.8%).
The driver is a **2.7x surge in trade volume** (793 vs 294 trades/day). Panic-driven, high-frequency trading creates dislocations that skilled traders can exploit.

### Insight 2 — Medium position sizing is the sweet spot
The Medium Size segment outperforms both Small and Large: **41% win rate** vs 22% (Small) and 33% (Large).
Small traders lack capital efficiency; large traders face adverse selection and fee drag.

### Insight 3 — Elite Traders are low-frequency, high-precision
Elite Traders average **only 66 trades/day** but generate **$365 PnL per trade** — 13x more than Scalpers ($28).
Frequency does not drive profitability. The highest-performing archetype trades least.

## 9. Strategy Recommendations

### Strategy 1 — Fear-Regime Contrarian Accumulation
**Rule:** When F&G < 35, scale into longs (+25-40% size) on confirmed support. Exit at F&G > 55.
**Target:** Elite Traders, Medium Size segment
**Risk:** Fear can cascade; use hard stops at -3% per trade. Win rate is lower so keep individual trade size controlled.

### Strategy 2 — Greed-Regime Precision Short-Selling
**Rule:** When F&G > 70, reduce long exposure and initiate short scalps on overbought setups. Exit all shorts if F&G > 80 for 3+ days.
**Target:** Scalpers (high-frequency, large-size)
**Risk:** Greed can persist; never fade momentum without a reversal signal. 1.5% hard stops.

## 10. Predictive Model — Next-Day Profitability

In [ ]:
ms = merged.sort_values('date').copy()
ms['next_pnl_pos'] = (ms['total_pnl'].shift(-1) > 0).astype(int)
mf_cols = ['value','trade_count','avg_size_usd','long_short_ratio','win_rate','pnl_std','rolling_pnl_7d']
mf = ms.dropna(subset=mf_cols+['next_pnl_pos'])

lr = LogisticRegression(max_iter=500, random_state=42)
cv = cross_val_score(lr, mf[mf_cols], mf['next_pnl_pos'],
                     cv=min(5, len(mf)//2), scoring='accuracy')
print(f"5-fold CV Accuracy: {cv.mean():.3f} +/- {cv.std():.3f}")
print(f"Per-fold: {cv.round(3)}")
print("Top features: sentiment index value, rolling 7d PnL, PnL volatility")

## 11. Save Outputs

In [ ]:
merged.to_csv("./cleaned_merged_data.csv", index=False)
print(f"Saved cleaned_merged_data.csv: {len(merged)} rows, {merged.shape[1]} columns")
print("All charts saved to ./outputs/")
print("Done.")